# Monte Carlo algorithm

*  Generate random densities $f_1,\dots,f_{n+1}$
* For each $t=1,\dots,n+1$:

* * simulate a process $(Z_{it}: i=1,\dots,n_t)$ with marginal density $f_t$ (possibly iid draws from $f_t$ for simplicity)
* * obtain the kernel density estimator $\hat{f}_t$ using the data $(Z_{it}: i=1,\dots,n_t)$
* * Define $Y_t = \Lambda(\hat{f}_t)$

* Apply the method of Bathia et al. (2010) to the sample $Y_1,\dots,Y_n$
* Obtain a forecast $\hat{Y}_{n+1}$
* Compare:
* * $\hat{Y}_{n+1}$ with $Y_{n+1}$
* * $\hat{Y}_{n+1}$ with $\Lambda(f_{n+1})$
* * $\hat{Y}_{n+1}$ with $X_{n+1}^{\text{boot}}$ (see definition below)
* * $\Lambda^{-1}(\hat{Y}_{n+1})$ with $\hat{f}_{n+1}$
* * $\Lambda^{-1}(\hat{Y}_{n+1})$ with ${f}_{n+1}$
* * $\Lambda^{-1}(\hat{Y}_{n+1})$ with $\Lambda^{-1}(X_{n+1}^{\text{boot}})$

To obtain $X_{t}^{\text{boot}}$:

* given $f_{t}$, for $b=1,\dots,B$,
* *    simulate a process $(Z_{it}^{(b)}: i=1,\dots,n_t)$ with marginal density $f_t$
* *    obtain the kernel density estimator $\hat{f}_t^{(b)}$ using the data $(Z_{it}^{(b)}: i=1,\dots,n_t)$
* *    define $Y_t^{(b)} = \Lambda(\hat{f}_t^{(b)})$
* * Define $X_t^{\text{boot}} = \frac1B \sum_{b=1}^B Y_{t}^{(b)}$


In [9]:
import numpy as np
from sklearn.model_selection import ParameterGrid

from pathlib import Path
from datetime import datetime
import joblib
from tqdm.auto import tqdm

import sys
sys.path.append('../../')
import src.forecasting.simulations      as sim
import src.fda.kde.estimators           as kde

In [2]:
EXECUTION_DATE : str = datetime.now().strftime('%Y%m%d')
LOG_PATH       : str = f'../logs/simulations/{EXECUTION_DATE}.log'
FILES_PATH     : str = f'../data/interim/simulation/{EXECUTION_DATE}/'
DATABASES_PATH : str = f'../data/interim/simulation/{EXECUTION_DATE}/databases/'
ALL_KDES_PATH  : str = f'../data/interim/simulation/{EXECUTION_DATE}/all_kdes.jbl'
CV_FILES_PATH  : str = f'../data/interim/simulation/{EXECUTION_DATE}/progress/'

Path(FILES_PATH).mkdir(parents=True, exist_ok=True)
Path(DATABASES_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_FILES_PATH).mkdir(parents=True, exist_ok=True)

In [3]:
# simulation objects
N_CURVES  = 300
N_REPS    = 100
N_RETURNS = 288 # (60/5)*12
m = 5001
x = np.linspace(-10, 10, m)
u = np.linspace(0, 1, m)

# base densities
pdf_normal = sim.generate_base_density(
                                       grid=x, 
                                       kind='gaussian',
                                       sigma=2, 
                                       mu=0
                                       )
pdf_t      = sim.generate_base_density(
                                       grid=x, 
                                       kind='student_t', 
                                       df=4, 
                                       scale=np.sqrt(2)
                                       )
base_densities = {
    "normal": pdf_normal,
    "t": pdf_t
}

# parameters
param_grid = {
    'base_pdf_name': list(base_densities.keys()),
    'basis': ['sine'],
    'dimensions': [2, 3, 4],
    'noise_type': ['Null'],
    'error_sigma': [0.25],
}

scenarios = list(ParameterGrid(param_grid))

scenarios = []
for i, params in enumerate(ParameterGrid(param_grid)):
    params['scenario_id'] = i
    scenarios.append(params)

In [10]:
# SIMULATE DATABASES
simulations_database = {}

n_scenarios = len(scenarios)

for scen_count, scenario in enumerate(scenarios, start=1):
    print(scenario)

    scenario_id = scenario["scenario_id"]
    pdf = base_densities[scenario['base_pdf_name']]

    simulations_database[scenario_id] = {
        "params": scenario.copy(),
        "replications": {}   # <-- FIX: now indexed by n_rep
    }

    for n_rep in range(N_REPS):
        print(f"\t{n_rep+1}/{N_REPS}")

        sim_engine = sim.FDFSimulator(u_grid=u, x_grid=x)

        sim_engine.run_simulation( 
            n_curves    = N_CURVES, 
            base_pdf    = pdf,
            basis       = scenario["basis"],   
            dimensions  = scenario["dimensions"],
            noise_type  = scenario["noise_type"],
            error_sigma = scenario["error_sigma"]
        )

        simulations_database[scenario_id]["replications"][n_rep] = {
            "densities": sim_engine.result["densities"],
            "samples":   sim_engine.get_samples_df(n_samples=N_RETURNS)
        }

    print(f"{scen_count}/{n_scenarios}")

{'base_pdf_name': 'normal', 'basis': 'sine', 'dimensions': 2, 'error_sigma': 0.25, 'noise_type': 'Null', 'scenario_id': 0}
	1/100
	2/100
	3/100
	4/100
	5/100
	6/100
	7/100
	8/100
	9/100
	10/100
	11/100
	12/100
	13/100
	14/100
	15/100
	16/100
	17/100
	18/100
	19/100
	20/100
	21/100
	22/100
	23/100
	24/100
	25/100
	26/100
	27/100
	28/100
	29/100
	30/100
	31/100
	32/100
	33/100
	34/100
	35/100
	36/100
	37/100
	38/100
	39/100
	40/100
	41/100
	42/100
	43/100
	44/100
	45/100
	46/100
	47/100
	48/100
	49/100
	50/100
	51/100
	52/100
	53/100
	54/100
	55/100
	56/100
	57/100
	58/100
	59/100
	60/100
	61/100
	62/100
	63/100
	64/100
	65/100
	66/100
	67/100
	68/100
	69/100
	70/100
	71/100
	72/100
	73/100
	74/100
	75/100
	76/100
	77/100
	78/100
	79/100
	80/100
	81/100
	82/100
	83/100
	84/100
	85/100
	86/100
	87/100
	88/100
	89/100
	90/100
	91/100
	92/100
	93/100
	94/100
	95/100
	96/100
	97/100
	98/100
	99/100
	100/100
1/6
{'base_pdf_name': 'normal', 'basis': 'sine', 'dimensions': 3, 'error_sigma': 0.25

In [4]:
# joblib.dump(simulations_database, f"{FILES_PATH}simulation_database.jbl")

# simulations_database = joblib.load(f"../data/interim/simulation/20260413/simulation_database.jbl")
simulations_database = joblib.load(f"{FILES_PATH}simulation_database.jbl")

In [7]:
# KDE params
# t_dfs = range(3,6)
t_dfs = [3]

rot_grid = {
    "method": ["rot"],
    "kernel": ["gaussian"],
    "sigma_robust": [False]
}

rot_grid_t = {
    "method": ["rot"],
    "kernel": ["t-student"],
    "df": [df for df in t_dfs],
    "sigma_robust": [False]
}

adaptive_grid = {
    "method": ["adaptive"], 
    "kernel": ["gaussian"]
}

adaptive_grid_t = {
    "method": ["adaptive"], 
    "kernel": ["t-student"],
    "df": [df for df in t_dfs],
}


density_param_grid = {}
for grid in [rot_grid, rot_grid_t, adaptive_grid, adaptive_grid_t]:
    for params in ParameterGrid(grid):
        
        key_parts = [params["kernel"]]
        if "method" in params: key_parts.append(params["method"])
        if "df" in params: key_parts.append(f"df={params['df']}")
        if params.get("sigma_robust"): key_parts.append("robust")
        if params.get("cv") == "LOO": key_parts.append("loo")
        
        full_model_name = "_".join(key_parts).replace(".", "")
        
        kernel_label = params["kernel"]
        if "df" in params:
            kernel_label += f"+df={params['df']}"
            
        bw_label = params.get("method", "fixed")
        if params.get("sigma_robust"): bw_label += "_robust"
        if params.get("cv") == "LOO": bw_label += "_loo"


        density_param_grid[full_model_name] = {
            "kernel": kernel_label,
            "bandwidth": bw_label,
            "params": params  
        }

print("KDE models:")
for name, values in density_param_grid.items():
    print("\t",name, ":", values["params"])

print(f"Total KDE models: {len(density_param_grid.items())}")

KDE models:
	 gaussian_rot : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}
	 t-student_rot_df=3 : {'df': 3, 'kernel': 't-student', 'method': 'rot', 'sigma_robust': False}
	 gaussian_adaptive : {'kernel': 'gaussian', 'method': 'adaptive'}
	 t-student_adaptive_df=3 : {'df': 3, 'kernel': 't-student', 'method': 'adaptive'}
Total KDE models: 4


In [11]:
density_param_grid

{'gaussian_rot': {'kernel': 'gaussian',
  'bandwidth': 'rot',
  'params': {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}},
 't-student_rot_df=3': {'kernel': 't-student+df=3',
  'bandwidth': 'rot',
  'params': {'df': 3,
   'kernel': 't-student',
   'method': 'rot',
   'sigma_robust': False}},
 'gaussian_adaptive': {'kernel': 'gaussian',
  'bandwidth': 'adaptive',
  'params': {'kernel': 'gaussian', 'method': 'adaptive'}},
 't-student_adaptive_df=3': {'kernel': 't-student+df=3',
  'bandwidth': 'adaptive',
  'params': {'df': 3, 'kernel': 't-student', 'method': 'adaptive'}}}

In [ ]:
total_models = len(density_param_grid) * N_REPS * len(scenarios)
pbar = tqdm(total=total_models, desc="Total CV Progress")

kde_databases = {}
for scenario, database in simulations_database.items():
    print(f"Processing configuration for {database['params']}")
    kde_databases[scenario] = {}
    sim_reps_database = simulations_database[scenario]["replications"]
    
    for n_rep, df in sim_reps_database.items():
        print(f"\t{n_rep}")
        returns_df = sim_reps_database[n_rep]['samples']
        kde_databases[scenario][n_rep] = {}
        
        for kde_bw_name, kde_bw_params in density_param_grid.items():
            print(f"\t simulation n. = {n_rep}: {kde_bw_name}")
            
            # bandwidths
            df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params["params"])    
            kde_params = {k: v for k, v in kde_bw_params["params"].items() if k in ['kernel', 'df']}
            
            # kdes
            df_grids, df_densities = kde.df_to_kde(
                X=returns_df, 
                h=df_h, 
                normalize_densities=False,
                **kde_params
            )

            rep_result = {
                        "scenario":     scenario,
                        "n_rep":        n_rep,
                        "model_name":   kde_bw_name,
                        "kde_params":   kde_bw_params["kernel"],
                        "kernel":       kde_bw_params["params"]["kernel"],
                        "bw_params":    kde_bw_params["bandwidth"],
                        "bw_method":    kde_bw_params["params"]["method"],
                        "df_h":         df_h,
                        "df_support":   df_grids,
                        "df_densities": df_densities
                }

            # kde_databases[scenario][n_rep][kde_bw_name] = rep_result
            
            joblib.dump(rep_result, f"{CV_FILES_PATH}scenario_{scenario}_rep_{n_rep}_kde_{kde_bw_name}.jbl")

            pbar.update(1)

            del df_h
            del df_grids
            del df_densities
            del kde_params


Total CV Progress:   0%|          | 0/2400 [00:00<?, ?it/s]

Processing configuration for {'base_pdf_name': 'normal', 'basis': 'sine', 'dimensions': 2, 'error_sigma': 0.25, 'noise_type': 'Null', 'scenario_id': 0}
	0
	 simulation n. = 0: gaussian_rot
	 simulation n. = 0: t-student_rot_df=3
